In [1]:
import os, json, math, random, re
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import pearsonr

# ----------------
# Repro / device
# ----------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ----------------
# Model choice (instruction-tuned E5)
# ----------------
E5_MODEL_NAME = "intfloat/multilingual-e5-large-instruct"

# ----------------
# Data source (DimABSA repo)
# ----------------
REPO_BASE = "https://raw.githubusercontent.com/DimABSA/DimABSA2026/main/task-dataset/track_a/subtask_1"
LANGUAGES = ["eng", "jpn", "rus", "ukr", "zho", "tat"]
DOMAINS   = ["laptop", "restaurant", "hotel", "device", "finance"]

# ----------------
# LLM guidance files (YOU set these paths)
# Expectation:
#  - arousal guidance CSV contains: ID,Aspect,(Language,Domain optional), llm_bin_bc, llm_p_low, llm_exp_score, llm_digit(optional)
#  - valence guidance CSV contains: ID,Aspect,(Language,Domain optional), llm_digit (1..8) OR llm_exp_score (optional)
# ----------------
VAL_GUIDE_TRAIN_CSV = "llm_preds_final_v2/valence_guidance_train.csv"
VAL_GUIDE_DEV_CSV   = "llm_preds_final_v2/valence_guidance_dev.csv"

ARO_GUIDE_TRAIN_CSV = "llm_preds_final_v2/arousal_guidance_train.csv"
ARO_GUIDE_DEV_CSV   = "llm_preds_final_v2/arousal_guidance_dev.csv"

# ----------------
# Training hyperparams (tuned for ~A100, ~1–2h target)
# ----------------
MAX_LEN = 256
BATCH_SIZE = 64          # if OOM: 32
GRAD_ACCUM = 1           # if you drop batch size, raise to keep effective batch
EPOCHS = 3               # can bump to 4 if time
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06

# Loss weights
LAMBDA_BIN_AUX = 0.15    # auxiliary coarse-bin classification (stabilizes)
HUBER_BETA = 0.5         # SmoothL1 beta

# Language balancing
USE_LANG_SAMPLER = True


/home/modim2/miniforge3/envs/hf-fa-py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


In [2]:
import requests

## === 1. DATA LOADING UTILITIES ===
def download_all_languages():
    """
    Crawls the repo and TAGS data with Language/Domain/Split for analysis.
    Handles '_alltasks' and '_task1' suffixes across train, dev, and test splits.
    """
    combined_data = []
    domains = ["laptop", "restaurant", "hotel", "device", "finance"] 
    suffixes = ["alltasks", "task1"]
    splits = ["train", "dev", "test"]  # Added splits
    
    print("\n--- Downloading Datasets ---")
    for lang in LANGUAGES:
        for domain in domains:
            for split in splits:
                found = False
                for suffix in suffixes:
                    # Modified filename to include dynamic {split} instead of hardcoded 'train'
                    filename = f"{lang}_{domain}_{split}_{suffix}.jsonl"
                    url = f"{REPO_BASE}/{lang}/{filename}"
                    
                    try:
                        resp = requests.get(url)
                        if resp.status_code == 200:
                            lines = resp.text.splitlines()
                            data = []
                            for line in lines:
                                row = json.loads(line)
                                row['Language'] = lang
                                row['Domain'] = domain
                                row['Split'] = split  # Tag the row with its split
                                data.append(row)
                            
                            print(f"✓ Loaded {lang}-{domain}-{split} (via {suffix}): {len(data)} samples")
                            combined_data.extend(data)
                            found = True
                            break # Stop checking suffixes if we found the file for this split
                    
                    except Exception as e:
                        print(f"Error fetching {url}: {e}")
                
                if not found:
                    # Optional: Print warning if a specific split is missing for a domain
                    # print(f"  x Missing {lang}-{domain}-{split}")
                    pass
                
    print(f"Total Combined Samples: {len(combined_data)}")
    return combined_data

def normalize_data(data):
    """
    Normalizes mixed JSONL formats into a standard DF.
    Handles 'Quadruplet' (Train), 'Aspect_VA' (Train/Dev), and 'Blind' formats (Test).
    """
    if not data:
        raise ValueError("No data downloaded! Check network or URLs.")
        
    print(f"Parsing Data from {len(data)} raw lines...")
    
    dfs = []
    
    # 1. Split data by type based on keys present
    quad_rows = [row for row in data if 'Quadruplet' in row]
    task1_rows = [row for row in data if 'Aspect_VA' in row]
    
    # Capture rows that are strictly meant for testing (marked as 'test' but missed above)
    test_rows = [
        row for row in data 
        if row['Split'] == 'test' 
        and 'Quadruplet' not in row 
        and 'Aspect_VA' not in row
    ]
    
    print(f" - Found {len(quad_rows)} Quadruplet rows")
    print(f" - Found {len(task1_rows)} Aspect_VA rows")
    print(f" - Found {len(test_rows)} Blind Test rows")

    # 2. Process Quadruplet Style
    if quad_rows:
        df_q = pd.json_normalize(
            quad_rows, 
            record_path='Quadruplet', 
            meta=['ID', 'Text', 'Language', 'Domain', 'Split']
        )
        # Parse VA: "5.5#4.0" -> [5.5, 4.0]
        df_q[['Valence', 'Arousal']] = df_q['VA'].str.split('#', expand=True).astype(float)
        dfs.append(df_q)
        
    # 3. Process Aspect_VA Style (Standard Task 1)
    if task1_rows:
        df_t = pd.json_normalize(
            task1_rows, 
            record_path='Aspect_VA', 
            meta=['ID', 'Text', 'Language', 'Domain', 'Split']
        )
        if 'Aspect' not in df_t.columns and 0 in df_t.columns:
             df_t = df_t.rename(columns={0: "Aspect"})
        
        # Parse VA
        df_t[['Valence', 'Arousal']] = df_t['VA'].str.split('#', expand=True).astype(float)
        dfs.append(df_t)

    # 4. Process Blind Test Data (Sample: {"Aspect": ["pizza", "sauce"], "Text": "...", "ID": "..."})
    if test_rows:
        try:
            df_test = pd.DataFrame(test_rows)
            
            # CRITICAL FIX: Explode the list of aspects into separate rows
            if 'Aspect' in df_test.columns:
                 df_test = df_test.explode('Aspect')
            
            # Add Dummy Columns for VA (Test set has no ground truth)
            # We set them to -1.0 to make it obvious they are dummies
            df_test['Valence'] = -1.0
            df_test['Arousal'] = -1.0
                
            dfs.append(df_test)
            
        except Exception as e:
            print(f"⚠️ Error parsing Test rows: {e}")

    if not dfs:
        raise ValueError("Unknown JSON format. No valid rows parsed.")
        
    # 5. Merge
    df = pd.concat(dfs, ignore_index=True)
    
    # Ensure all required columns exist
    required_cols = ['ID', 'Text', 'Aspect', 'Valence', 'Arousal', 'Language', 'Domain', 'Split']
    for col in required_cols:
        if col not in df.columns:
            df[col] = np.nan 

    return df[required_cols].copy()

In [3]:
def clean_aspect(x):
    if pd.isna(x) or str(x).upper() == "NULL":
        return "general sentiment"
    return str(x)

raw = download_all_languages()
df = normalize_data(raw)
df["Aspect"] = df["Aspect"].apply(clean_aspect)

train_source_df  = df[df["Split"] == "train"].copy()
official_dev_df  = df[df["Split"] == "dev"].copy()
official_test_df = df[df["Split"] == "test"].copy()

print("train:", len(train_source_df), "dev:", len(official_dev_df), "test:", len(official_test_df))


--- Downloading Datasets ---
✓ Loaded eng-laptop-train (via alltasks): 4076 samples
✓ Loaded eng-laptop-dev (via task1): 200 samples
✓ Loaded eng-laptop-test (via task1): 1000 samples
✓ Loaded eng-restaurant-train (via alltasks): 2284 samples
✓ Loaded eng-restaurant-dev (via task1): 200 samples
✓ Loaded eng-restaurant-test (via task1): 1000 samples
✓ Loaded jpn-hotel-train (via alltasks): 1600 samples
✓ Loaded jpn-hotel-dev (via task1): 200 samples
✓ Loaded jpn-hotel-test (via task1): 800 samples
✓ Loaded jpn-finance-train (via task1): 1024 samples
✓ Loaded jpn-finance-dev (via task1): 200 samples
✓ Loaded jpn-finance-test (via task1): 800 samples
✓ Loaded rus-restaurant-train (via alltasks): 1240 samples
✓ Loaded rus-restaurant-dev (via task1): 56 samples
✓ Loaded rus-restaurant-test (via task1): 1072 samples
✓ Loaded ukr-restaurant-train (via alltasks): 1240 samples
✓ Loaded ukr-restaurant-dev (via task1): 56 samples
✓ Loaded ukr-restaurant-test (via task1): 1072 samples
✓ Loaded zh

In [4]:
# ----------------------------
# Load guidance CSVs (train+dev)
# ----------------------------
def _load_csv(path, name):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {name}: {path}")
    df = pd.read_csv(path)
    for c in ["ID", "Aspect"]:
        if c not in df.columns:
            raise ValueError(f"{name} missing required column '{c}'")
    return df

val_g_train = _load_csv(VAL_GUIDE_TRAIN_CSV, "VAL_GUIDE_TRAIN_CSV")
val_g_dev   = _load_csv(VAL_GUIDE_DEV_CSV,   "VAL_GUIDE_DEV_CSV")
aro_g_train = _load_csv(ARO_GUIDE_TRAIN_CSV, "ARO_GUIDE_TRAIN_CSV")
aro_g_dev   = _load_csv(ARO_GUIDE_DEV_CSV,   "ARO_GUIDE_DEV_CSV")

val_g = pd.concat([val_g_train, val_g_dev], ignore_index=True)
aro_g = pd.concat([aro_g_train, aro_g_dev], ignore_index=True)

# Drop duplicates by (ID, Aspect) (keep first)
val_g = val_g.drop_duplicates(subset=["ID", "Aspect"], keep="first")
aro_g = aro_g.drop_duplicates(subset=["ID", "Aspect"], keep="first")

# ----------------------------
# Sanity checks / required columns
# ----------------------------
# Valence: you showed these columns exist (great)
VAL_EXPECTED = ["llm_digit","llm_bin_abc","llm_pA","llm_pB","llm_pC","llm_conf","llm_exp_score"]
missing_val = [c for c in VAL_EXPECTED if c not in val_g.columns]
if missing_val:
    raise ValueError(f"Valence guidance missing columns: {missing_val}")

# Arousal: minimum required for your current pipeline
ARO_EXPECTED = ["llm_bin_bc","llm_p_low","llm_exp_score"]
missing_aro = [c for c in ARO_EXPECTED if c not in aro_g.columns]
if missing_aro:
    raise ValueError(f"Arousal guidance missing columns: {missing_aro}")

# ----------------------------
# Combine official train + official dev as labeled
# ----------------------------
labeled = pd.concat([train_source_df, official_dev_df], ignore_index=True)
labeled = labeled[(labeled["Valence"] >= 0) & (labeled["Arousal"] >= 0)].copy()

# ----------------------------
# Merge valence guidance (ALL useful columns)
# ----------------------------
val_keep = ["ID","Aspect"] + VAL_EXPECTED
labeled = labeled.merge(val_g[val_keep], on=["ID","Aspect"], how="left")

# ----------------------------
# Merge arousal guidance (required columns + optional extras if present)
# ----------------------------
aro_optional = [c for c in ["llm_digit","llm_dist_str"] if c in aro_g.columns]
aro_keep = ["ID","Aspect"] + ARO_EXPECTED + aro_optional

# Use suffix to avoid clobbering valence llm_* columns if any overlap
labeled = labeled.merge(
    aro_g[aro_keep],
    on=["ID","Aspect"],
    how="left",
    suffixes=("", "_aro")
)

# ----------------------------
# Fill missing guidance values defensively
# ----------------------------
# Valence defaults
labeled["llm_bin_abc"] = labeled["llm_bin_abc"].fillna("[BIN_UNK]")
labeled["llm_digit"]   = labeled["llm_digit"].fillna(5).astype(float)

for c in ["llm_pA","llm_pB","llm_pC","llm_conf"]:
    labeled[c] = labeled[c].fillna(0.0).astype(float)

labeled["llm_exp_score"] = labeled["llm_exp_score"].fillna(5.0).astype(float)

# Arousal defaults (these are arousal columns: llm_bin_bc/llm_p_low/llm_exp_score)
labeled["llm_bin_bc"] = labeled["llm_bin_bc"].fillna("[BIN_UNK]")
labeled["llm_p_low"]  = labeled["llm_p_low"].fillna(0.0).astype(float)

# NOTE: arousal guidance exp_score column name overlaps "llm_exp_score" from valence if you didn't suffix.
# In the merge above, we used suffixes=("", "_aro"), but since arousal exp_score is also named llm_exp_score,
# it will become "llm_exp_score_aro" IF valence already had llm_exp_score.
if "llm_exp_score_aro" in labeled.columns:
    labeled["llm_exp_score_aro"] = labeled["llm_exp_score_aro"].fillna(6.0).astype(float)
else:
    # If you didn't have valence exp_score somehow, it may stay as llm_exp_score
    labeled["llm_exp_score"] = labeled["llm_exp_score"].fillna(6.0).astype(float)

print("Merged labeled rows:", len(labeled))
print("Missing valence guidance rate:",
      float((labeled["llm_bin_abc"] == "[BIN_UNK]").mean()))
print("Missing arousal guidance rate:",
      float((labeled["llm_bin_bc"] == "[BIN_UNK]").mean()))


Merged labeled rows: 42336
Missing valence guidance rate: 0.0
Missing arousal guidance rate: 0.0


In [5]:
from sklearn.model_selection import train_test_split
from collections import Counter

train_df, test_df = train_test_split(labeled, test_size=0.10, random_state=SEED, shuffle=True)

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))

def make_language_weights(df_in):
    langs = df_in["Language"].tolist()
    counts = Counter(langs)
    w = np.array([1.0 / counts[l] for l in langs], dtype=np.float32)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)

train_sampler = None
if USE_LANG_SAMPLER:
    weights = make_language_weights(train_df)
    train_sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    print("Using language-balanced sampler.")


Train rows: 38102
Test rows : 4234
Using language-balanced sampler.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(E5_MODEL_NAME, use_fast=True)

def bucket_p_low(p):
    # discretize p_low into coarse buckets to make it easier for the encoder
    # (keeps it robust; pure floats in text can be noisy)
    p = float(p)
    if p < 0.05: return "p0"
    if p < 0.15: return "p1"
    if p < 0.35: return "p2"
    if p < 0.65: return "p3"
    return "p4"

def round_exp(x):
    # arousal exp_score is ~3..8 ; valence exp might be 1..8 if you have it
    try:
        return f"{float(x):.2f}"
    except:
        return "UNK"

def bucket01(x):
    # bucket a 0..1 number to a small discrete alphabet (robust + easier to learn)
    x = float(x)
    if x < 0.05: return "v0"
    if x < 0.15: return "v1"
    if x < 0.30: return "v2"
    if x < 0.50: return "v3"
    if x < 0.70: return "v4"
    if x < 0.85: return "v5"
    return "v6"

def round2(x, default="UNK"):
    try:
        return f"{float(x):.2f}"
    except:
        return default

def safe_int(x, default="UNK"):
    try:
        return str(int(float(x)))
    except:
        return default


# def build_e5_input(row):
#     """
#     Instruction-style single input string.
#     Uses ALL valence guidance: bin, digit, exp, conf, pA/pB/pC.
#     Uses arousal guidance: bin_bc, p_low, exp_score.
#     """
#     # ---- Valence guidance
#     vbin = row.get("llm_bin_abc", "[BIN_UNK]")
#     vdig = safe_int(row.get("llm_digit", None))
#     vexp = round2(row.get("llm_exp_score", 6.0))
#     vconf = bucket01(row.get("llm_conf", 0.0))
#     vpA = bucket01(row.get("llm_pA", 0.0))
#     vpB = bucket01(row.get("llm_pB", 0.0))
#     vpC = bucket01(row.get("llm_pC", 0.0))

#     # ---- Arousal guidance
#     abin = row.get("llm_bin_bc", "[BIN_UNK]")
#     aplow = bucket01(row.get("llm_p_low", 0.0))
#     aexp = round2(row.get("llm_exp_score_aro", 6.0))
    
#     instruction = (
#         "Predict valence and arousal (continuous) toward the aspect. "
#         "Use the guidance features if helpful, but rely on the text."
#     )

#     # Keep guidance compact and discrete-ish
#     guide = (
#         f"V:bin={vbin} digit={vdig} exp={vexp} conf={vconf} pA={vpA} pB={vpB} pC={vpC} | "
#         f"A:bin={abin} pLow={aplow} exp={aexp}"
#     )

#     query = (
#         f"Lang={row['Language']} Domain={row['Domain']} "
#         f"{guide} "
#         f"Aspect: {row['Aspect']} "
#         f"Text: {row['Text']}"
#     )

#     return f"Instruct: {instruction}\nQuery: {query}"

# def build_e5_input(row):
#     """
#     Single-string instruction/query format for multilingual-e5-large-instruct.
#     """
#     vbin = row.get("llm_bin_abc", "[BIN_UNK]")
#     abin = row.get("llm_bin_bc", "[BIN_UNK]")
#     plow = bucket_p_low(row.get("llm_p_low", 0.0))
#     aexp = round_exp(row.get("llm_exp_score", 6.0))

#     # If you also have valence llm_digit, you can add it as a discrete hint:
#     vdig = row.get("llm_digit", None)
#     if vdig is not None and not pd.isna(vdig):
#         vdig = str(int(float(vdig)))
#     else:
#         vdig = "UNK"

#     instruction = (
#         "Predict the valence and arousal (continuous 1-9 scale) expressed toward the given aspect in the text. "
#         "Use the optional coarse guidance tokens if helpful."
#     )

#     query = (
#         f"Lang={row['Language']} Domain={row['Domain']} "
#         f"V_GUIDE={vbin} V_DIGIT={vdig} "
#         f"A_GUIDE={abin} A_PLOW={plow} A_EXP={aexp} "
#         f"Aspect: {row['Aspect']} "
#         f"Text: {row['Text']}"
#     )

#     return f"Instruct: {instruction}\nQuery: {query}"


In [7]:
class VADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        x = build_e5_input(r)

        enc = self.tokenizer(
            x,
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors="pt"
        )

        v = float(r["Valence"])
        a = float(r["Arousal"])

        # coarse bins for auxiliary loss
        def bin_abc(y):
            if y < 3.0: return 0
            if y < 6.0: return 1
            return 2

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "valence": torch.tensor(v, dtype=torch.float32),
            "arousal": torch.tensor(a, dtype=torch.float32),
            "v_bin": torch.tensor(bin_abc(v), dtype=torch.long),
            "a_bin": torch.tensor(bin_abc(a), dtype=torch.long),
            "language": r["Language"],
        }

def collate_fn(batch):
    input_ids = [b["input_ids"] for b in batch]
    attention_mask = [b["attention_mask"] for b in batch]
    enc = tokenizer.pad(
        {"input_ids": input_ids, "attention_mask": attention_mask},
        padding=True,
        return_tensors="pt"
    )

    out = {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "valence": torch.stack([b["valence"] for b in batch]),
        "arousal": torch.stack([b["arousal"] for b in batch]),
        "v_bin": torch.stack([b["v_bin"] for b in batch]),
        "a_bin": torch.stack([b["a_bin"] for b in batch]),
        "language": [b["language"] for b in batch],
    }
    return out

train_ds = VADataset(train_df, tokenizer, max_len=MAX_LEN)
test_ds  = VADataset(test_df, tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=train_sampler if train_sampler is not None else None,
    shuffle=False if train_sampler is not None else True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

print("train batches:", len(train_loader), "test batches:", len(test_loader))


train batches: 596 test batches: 67


In [8]:
print(build_e5_input(train_df.iloc[0]))

Instruct: Predict valence and arousal (continuous) toward the aspect. Use the guidance features if helpful, but rely on the text.
Query: Lang=jpn Domain=hotel V:bin=[BIN_C] digit=8 exp=7.91 conf=v6 pA=v0 pB=v0 pC=v6 | A:bin=[BIN_C] pLow=v0 exp=6.92 Aspect: 食事 Text: 食事はもちろんスタッフの対応も素晴らしかったです


In [9]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class E5MultiTaskVA(nn.Module):
    def __init__(self, backbone_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone_name)
        hidden = self.backbone.config.hidden_size

        # light regularization
        self.dropout = nn.Dropout(0.15)

        # shared projection (helps transfer across languages/domains)
        self.proj = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        # regression heads
        self.val_head = nn.Linear(hidden, 1)
        self.aro_head = nn.Linear(hidden, 1)

        # auxiliary coarse-bin heads (A/B/C)
        self.val_bin_head = nn.Linear(hidden, 3)
        self.aro_bin_head = nn.Linear(hidden, 3)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(out.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        z = self.proj(pooled)

        val = self.val_head(z).squeeze(-1)
        aro = self.aro_head(z).squeeze(-1)

        vbin_logits = self.val_bin_head(z)
        abin_logits = self.aro_bin_head(z)

        # constrain outputs to [1,9] like your previous regression trick
        val = 1.0 + 8.0 * torch.sigmoid(val)
        aro = 1.0 + 8.0 * torch.sigmoid(aro)

        return val, aro, vbin_logits, abin_logits

model = E5MultiTaskVA(E5_MODEL_NAME).to(device)


In [10]:
from torch.amp import autocast, GradScaler

reg_crit = nn.SmoothL1Loss(beta=HUBER_BETA)
bin_crit = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = (len(train_loader) * EPOCHS) // GRAD_ACCUM
warmup_steps = int(WARMUP_FRAC * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = GradScaler(enabled=(device.type == "cuda"))

def clip_and_round_va(v, a):
    """
    Clip to [1, 9] and round to 2 decimals, as required by the task.
    v, a can be numpy arrays or python lists.
    """
    v = np.asarray(v, dtype=np.float64)
    a = np.asarray(a, dtype=np.float64)
    v = np.clip(v, 1.0, 9.0)
    a = np.clip(a, 1.0, 9.0)
    v = np.round(v, 2)
    a = np.round(a, 2)
    return v, a

def compute_metrics(pred_v, gold_v, pred_a, gold_a):
    pred_v = np.asarray(pred_v, dtype=np.float64)
    pred_a = np.asarray(pred_a, dtype=np.float64)
    gold_v = np.asarray(gold_v, dtype=np.float64)
    gold_a = np.asarray(gold_a, dtype=np.float64)

    # Apply task constraints BEFORE metric:
    pred_v, pred_a = clip_and_round_va(pred_v, pred_a)

    # Official RMSE_VA:
    mse_va = np.mean((pred_v - gold_v) ** 2 + (pred_a - gold_a) ** 2)
    rmse_va = float(np.sqrt(mse_va))

    # (Optional extra logging)
    rmse_v = float(np.sqrt(np.mean((pred_v - gold_v) ** 2)))
    rmse_a = float(np.sqrt(np.mean((pred_a - gold_a) ** 2)))

    # Pearson (optional)
    def safe_pcc(x, y):
        if np.std(x) < 1e-8 or np.std(y) < 1e-8:
            return 0.0
        return float(pearsonr(x, y)[0])

    pcc_v = safe_pcc(pred_v, gold_v)
    pcc_a = safe_pcc(pred_a, gold_a)

    return rmse_va, rmse_v, rmse_a, pcc_v, pcc_a

@torch.no_grad()
def evaluate(loader):
    model.eval()
    pv, gv, pa, ga = [], [], [], []

    for batch in tqdm(loader, desc="eval", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        gold_v = batch["valence"].to(device)
        gold_a = batch["arousal"].to(device)

        pred_v, pred_a, _, _ = model(input_ids, attention_mask)

        pv.extend(pred_v.detach().cpu().tolist())
        pa.extend(pred_a.detach().cpu().tolist())
        gv.extend(gold_v.detach().cpu().tolist())
        ga.extend(gold_a.detach().cpu().tolist())

    return compute_metrics(pv, gv, pa, ga)


best = 1e9
# os.makedirs("models_va", exist_ok=True)
BEST_PATH = "models_va_v1/best_e5_va.pt"

# for epoch in range(1, EPOCHS+1):
#     model.train()
#     optimizer.zero_grad(set_to_none=True)

#     running = 0.0
#     step = 0

#     for i, batch in enumerate(tqdm(train_loader, desc=f"train epoch {epoch}")):
#         input_ids = batch["input_ids"].to(device, non_blocking=True)
#         attention_mask = batch["attention_mask"].to(device, non_blocking=True)
#         gold_v = batch["valence"].to(device, non_blocking=True)
#         gold_a = batch["arousal"].to(device, non_blocking=True)
#         vbin = batch["v_bin"].to(device, non_blocking=True)
#         abin = batch["a_bin"].to(device, non_blocking=True)

#         with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type=="cuda")):
#             pred_v, pred_a, vbin_logits, abin_logits = model(input_ids, attention_mask)

#             loss_v = reg_crit(pred_v, gold_v)
#             loss_a = reg_crit(pred_a, gold_a)
#             loss = loss_v + loss_a

#             if LAMBDA_BIN_AUX > 0:
#                 loss += LAMBDA_BIN_AUX * (bin_crit(vbin_logits, vbin) + bin_crit(abin_logits, abin))

#             loss = loss / GRAD_ACCUM

#         scaler.scale(loss).backward()

#         if (i + 1) % GRAD_ACCUM == 0:
#             scaler.step(optimizer)
#             scaler.update()
#             optimizer.zero_grad(set_to_none=True)
#             scheduler.step()

#         running += float(loss.detach().cpu())
#         step += 1

#     rmse_va, rmse_v, rmse_a, pcc_v, pcc_a = evaluate(test_loader)

#     # Select best by the official metric
#     score = rmse_va

#     print(f"\nEpoch {epoch}: train_loss={running/step:.4f} | "
#         f"TEST rmse_va={rmse_va:.4f} | rmse_v={rmse_v:.4f} rmse_a={rmse_a:.4f} | "
#         f"pcc_v={pcc_v:.3f} pcc_a={pcc_a:.3f}")

#     if score < best:
#         best = score
#         torch.save(model.state_dict(), BEST_PATH)
#         print("  ✅ saved best ->", BEST_PATH)



In [12]:
# Load best
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()

@torch.no_grad()
def predict_dataframe(df_in, batch_size=64):
    ds = VADataset(df_in, tokenizer, max_len=MAX_LEN)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

    pv_all, pa_all = [], []

    for batch in tqdm(loader, desc="predict"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        pv, pa, _, _ = model(input_ids, attention_mask)
        pv_all.extend(pv.detach().cpu().tolist())
        pa_all.extend(pa.detach().cpu().tolist())

    # Apply task constraints
    pv_all, pa_all = clip_and_round_va(pv_all, pa_all)

    out = df_in.reset_index(drop=True).copy()
    out["pred_valence"] = pv_all
    out["pred_arousal"] = pa_all
    return out

# For test set, you still want to include guidance merges.
# So build a test_df_with_guidance similarly (ID, Aspect merge) before calling predict_dataframe.

test_for_submit = official_test_df.copy()

# merge guidance for test (from your already-generated guidance CSVs)
VAL_GUIDE_TEST_CSV = "llm_preds_final_v2/valence_guidance_test.csv"
ARO_GUIDE_TEST_CSV = "llm_preds_final_v2/arousal_guidance_test.csv"

val_g_test = pd.read_csv(VAL_GUIDE_TEST_CSV).drop_duplicates(subset=["ID","Aspect"], keep="first")
aro_g_test = pd.read_csv(ARO_GUIDE_TEST_CSV).drop_duplicates(subset=["ID","Aspect"], keep="first")
# val_g_test = ensure_val_guidance_cols(val_g_test)
# aro_g_test = ensure_aro_guidance_cols(aro_g_test)

test_for_submit = test_for_submit.merge(val_g_test[["ID","Aspect","llm_bin_abc"] + ([c for c in ["llm_digit","llm_exp_score","llm_dist_str"] if c in val_g_test.columns])],
                                        on=["ID","Aspect"], how="left")
test_for_submit = test_for_submit.merge(aro_g_test[["ID","Aspect","llm_bin_bc","llm_p_low","llm_exp_score"] + ([c for c in ["llm_digit","llm_dist_str"] if c in aro_g_test.columns])],
                                        on=["ID","Aspect"], how="left", suffixes=("", "_aro"))

test_for_submit["llm_bin_abc"] = test_for_submit["llm_bin_abc"].fillna("[BIN_UNK]")
test_for_submit["llm_bin_bc"]  = test_for_submit["llm_bin_bc"].fillna("[BIN_UNK]")
test_for_submit["llm_p_low"]   = test_for_submit["llm_p_low"].fillna(0.0)
test_for_submit["llm_exp_score"] = test_for_submit["llm_exp_score"].fillna(6.0)

pred_test = predict_dataframe(test_for_submit, batch_size=BATCH_SIZE)

pred_test.to_csv("predictions_va_e5_instruct.csv", index=False)
print("Wrote predictions_va_e5_instruct.csv")

/tmp/ipykernel_2286193/2011069584.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(BEST_PATH, map_location=device))
predict:   0%|       

Wrote predictions_va_e5_instruct.csv
